# 🧠 LLM Fundamentals — Basics
### Frameworks, Gemini setup, parameters, memory, and RAG concepts
**Stack:** Python + LangChain + Google Gemini

This notebook covers the fundamentals only. The actual Policy RAG Chatbot project lives in **`RAG_Based_ChatBot.ipynb`**.

### Roadmap
1. Introduction to AI Frameworks
2. Configure the LLM (Gemini via LangChain)
3. LLM Parameters
4. Build a Basic Chatbot
5. Conversation History & Memory
6. Introduce RAG

## 1. Introduction to AI Frameworks

A **framework** (e.g. LangChain) sits on top of raw LLM API calls and gives reusable building blocks: prompts, memory, retrieval (RAG), tools/agents — with adapters for almost any model provider.

We'll use **Python + LangChain** here since it's the most mature/documented option and needs zero local setup.

## 2. Configure the LLM

### 2.1 Install packages

In [ ]:
# LangChain core + the Gemini integration + Google's own SDK (used for a couple of direct calls later)
# pypdf/langchain-community: PDF loading. gradio: the demo UI for the Policy RAG Chatbot.
!pip install -q langchain langchain-core langchain-community langchain-google-genai google-generativeai langchain-text-splitters pypdf gradio

### 2.2 Provide your Gemini API key

We use `getpass` so the key is never echoed or saved into the notebook file — get one at **[Google AI Studio](https://aistudio.google.com/app/apikey)**.

In [ ]:
import os
from getpass import getpass

# You will be prompted to paste your key below. It will NOT be displayed or saved to the notebook.
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

print("API key loaded into this session ✅ (not printed, not saved to the notebook file)")


API key loaded into this session ✅ (not printed, not saved to the notebook file)


### 2.3 Initialize the Gemini model through LangChain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",  # a fast, low-cost Gemini model — good default for a workshop
    temperature=0.7,                # we'll explain this in Section 3
)

print("Model initialized:", llm.model)


Model initialized: gemini-3.1-flash-lite


### 2.4 Your first LLM call


In [ ]:
response = llm.invoke("In one sentence, explain what a Large Language Model is.")
print(response.content)


[{'type': 'text', 'text': 'A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.', 'extras': {'signature': 'EnEKbwERTTIP3tnOg2Bx+r8bGjGRHwP+TagQGIgt3CDMY98arhba5rQ8P7kBUdwUeoXNXR6ZWyVhZQnfXi1fbgyC82vJZNW880Nz3VYht070m+eni/i7s/SPU6V+8R3dZECOns+6iIMnGC/UeAJpl3oLUQ=='}}]


`response.content` can come back as a plain string or a list of content blocks. Either way we only want the visible text, so here's a small helper we'll reuse for the rest of the notebook.

In [ ]:
def get_text(response):
    """Return just the plain text of an LLM response, whether `.content` is a string
    or a list of content blocks (e.g. text + an internal thought-signature block)."""
    content = response.content
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block, dict) and block.get("type") == "text"
    )

print(get_text(response))


A Large Language Model is a type of artificial intelligence trained on vast amounts of text data to understand, generate, and predict human language patterns.


**Important:** Gemini has no memory between calls — that's why Section 5 exists.

## 3. LLM Parameters

| Parameter | What it controls |
|---|---|
| `temperature` | Randomness of output (`0.0` deterministic → `1.0`+ more random) |
| `max_output_tokens` | Hard cap on response length |
| `model` | Speed/cost vs. capability |

### 3.1 Temperature: low vs. high, same prompt

In [ ]:
prompt = "Give me a one-sentence tagline for a coffee shop."

llm_low_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)
llm_high_temp = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=1.0)

print("=== temperature = 0.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_low_temp.invoke(prompt)))

print("\n=== temperature = 1.0 (run 3 times) ===")
for i in range(3):
    print(f"{i+1}.", get_text(llm_high_temp.invoke(prompt)))


=== temperature = 0.0 (run 3 times) ===
1. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
2. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*   **For a high-energy vibe:** "Fueling your ambition, one pour at a time."
*   **For a quality-focused vibe:** "Crafted with passion, poured to perfection."
*   **For a community-focused vibe:** "Where great coffee meets good company."
*   **Short and punchy:** "Awaken your senses."
3. Here are a few options, depending on the vibe of your shop:

*   **For a cozy vibe:** "Your daily escape, one cup at a time."
*

Low temperature → nearly identical runs. High temperature → more variety.

### 3.2 Maximum output tokens

In [ ]:
prompt = "Explain how a vector database works."

llm_short = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=20)
llm_long = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.3, max_output_tokens=300)

print("=== max_output_tokens = 20 ===")
print(get_text(llm_short.invoke(prompt)))

print("\n=== max_output_tokens = 300 ===")
print(get_text(llm_long.invoke(prompt)))


=== max_output_tokens = 20 ===
To understand how a vector database works, you first need to understand the concept of

=== max_output_tokens = 300 ===
To understand how a vector database works, you first have to understand the core concept: **Vector Embeddings.**

In traditional databases (SQL), you search for data using exact matches (e.g., "Find the user with ID 123"). In a vector database, you search for **meaning** (e.g., "Find documents that are conceptually similar to this query").

Here is the step-by-step breakdown of how a vector database functions.

---

### 1. The Foundation: Vector Embeddings
Computers cannot understand the meaning of a photo, a paragraph of text, or an audio file. To bridge this gap, we use **Machine Learning models** (like OpenAI’s `text-embedding-3` or CLIP for images) to convert raw data into a **Vector Embedding**.

*   **What is a vector?** It is simply a long list of numbers (e.g., `[0.12, -0.59, 0.88, ...]`).
*   **The Magic:** These numbers represe

### 3.3 Model selection

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Models available to your key that support chat:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(" -", m.name)


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Models available to your key that support chat:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - mode

Comparing `gemini-3.1-flash-lite` vs `gemini-3.5-flash` on the same prompt.

In [ ]:
import time

prompt = "Explain the difference between Agentic AI and RAG in 2 sentences."

for model_name in ["gemini-3.1-flash-lite", "gemini-3.5-flash"]:
    fast_llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.3)
    start = time.time()
    result = fast_llm.invoke(prompt)
    elapsed = time.time() - start
    print(f"--- {model_name} ({elapsed:.1f}s) ---")
    print(get_text(result))
    print()


--- gemini-3.1-flash-lite (8.4s) ---
RAG (Retrieval-Augmented Generation) is a technique that provides an AI with external data to improve the accuracy of its responses to specific queries. In contrast, Agentic AI refers to autonomous systems capable of using tools, reasoning through multi-step plans, and taking independent actions to achieve complex goals.

--- gemini-3.5-flash (10.0s) ---
**RAG (Retrieval-Augmented Generation)** is a technique that improves an AI's answers by fetching relevant external data to ground its responses in specific facts. In contrast, **Agentic AI** is an autonomous system designed to proactively plan, make decisions, use tools, and execute multi-step workflows to achieve complex goals.



**Note:** faster model = quicker, less nuanced answer; larger model = slower, more thorough. Pick based on latency/cost/task needs.

## 4. Build a Basic Chatbot

`User → LLM → Response` — no memory, every message is a fresh call.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.7)

def chat(user_message):
    response = llm.invoke(user_message)
    return get_text(response)

print(chat("Hi! My name is Alex."))


Hi Alex! It’s great to meet you. How are you doing today? Is there anything I can help you with?


In [ ]:
print(chat("What is my name?"))


I don’t know your name. As an AI, I don’t have access to your personal identity, documents, or private information unless you have previously shared it with me in this specific conversation.


The model doesn't know your name — each `chat()` call is stateless and independent. Real chat products resend the conversation each time, which is what we build next.

## 5. Conversation History & Memory

The application resends the growing message list on every call. LangChain represents it as typed messages: `SystemMessage`, `HumanMessage`, `AIMessage`.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

history = [SystemMessage(content="You are a friendly, concise assistant.")]

def chat_with_history(user_message):
    history.append(HumanMessage(content=user_message))
    response = llm.invoke(history)          # send the WHOLE conversation so far
    reply_text = get_text(response)
    history.append(AIMessage(content=reply_text))
    return reply_text

print(chat_with_history("My name is Alex."))


Hi Alex! It's nice to meet you. How can I help you today?


In [ ]:
print(chat_with_history("What is my name?"))


Your name is Alex!


Now it answers "Alex" — the full `history` list is resent every call. Here's what's actually sent:

In [ ]:
for msg in history:
    print(f"[{msg.type}] {msg.content}")


[system] You are a friendly, concise assistant.
[human] My name is Alex.
[ai] Hi Alex! It's nice to meet you. How can I help you today?
[human] What is my name?
[ai] Your name is Alex!


Longer conversations mean more resent text each turn — real products eventually trim or summarize older history.

## 6. Introduce RAG (Retrieval-Augmented Generation)

Instead of pasting a whole document into the prompt (context window limits, cost, distraction), RAG searches it first and sends only the relevant pieces:

```
Document → Load → Chunk → Embedding → Vector Database → Similarity Search → Relevant Chunks → LLM → Answer
```